# 6. Bootstrap Confidence Intervals
Works on **any dataset**. A **file explorer window** opens in Step 2. Columns for each analysis are **auto-selected** (printed in Step 3) - override them in the CONFIG cell if your question names specific columns.

**Covers:** mean, median, std, trimmed mean, proportion, difference of two groups, correlation, regression slope, any custom statistic | 4 CI methods: **Percentile, Basic, Normal, BCa** | comparison with classical intervals | convergence in B | coverage simulation

## Step 0: Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

## Step 1: CONFIG (leave as None for automatic selection)

In [ ]:
CONF_LEVEL = 0.95        # confidence level (0.90, 0.95, 0.99 ...)
B          = 10000       # number of bootstrap resamples
SEED       = 42          # for reproducible results
TRIM       = 0.10        # proportion trimmed from each tail for the trimmed mean

NUM_COL    = None        # numeric column for mean / median / std, e.g. 'age'

CAT_COL    = None        # categorical column for the proportion, e.g. 'gender'
POS_LEVEL  = None        # the category whose proportion you want, e.g. 'Female'

GROUP_COL    = None      # categorical column defining 2 groups, e.g. 'smoker'
GROUP_LEVELS = None      # e.g. ('yes', 'no'). None = two largest groups
GROUP_NUM    = None      # numeric column compared between groups. None = NUM_COL

CORR_X     = None        # numeric column x (also the predictor in the regression slope)
CORR_Y     = None        # numeric column y

CUSTOM_STAT = None       # any function of a 1-D array, e.g. lambda a: np.percentile(a, 90)
                         # None = coefficient of variation (std/mean) as a demo

RUN_COVERAGE = True      # coverage simulation (Step 12) - takes a few seconds

## Step 2: Select Dataset from File Explorer

In [ ]:
def pick_file():
    """Opens the OS file explorer. Falls back to typing the path if no GUI is available."""
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        path = filedialog.askopenfilename(
            title='Select your dataset',
            filetypes=[('Data files', '*.csv *.xlsx *.xls *.json *.txt *.tsv'), ('All files', '*.*')])
        root.destroy()
    except Exception:
        print('File dialog not available -> type the path instead.')
        path = input('Enter full file path: ')
    return path.strip().strip('"').strip("'")

def load_data(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.xlsx', '.xls'):
        return pd.read_excel(path)
    if ext == '.json':
        return pd.read_json(path)
    try:
        return pd.read_csv(path, sep=None, engine='python')
    except UnicodeDecodeError:
        return pd.read_csv(path, sep=None, engine='python', encoding='latin-1')

file_path = pick_file()
if not file_path:
    raise SystemExit('No file selected. Re-run this cell.')

df = load_data(file_path)
df.columns = df.columns.astype(str).str.strip()
print('Loaded:', file_path, '| Shape:', df.shape)
df.head()

## Step 3: Light Cleaning + Automatic Column Selection

In [ ]:
def text_cols(d):
    return [col for col in d.columns
            if pd.api.types.is_object_dtype(d[col]) or pd.api.types.is_string_dtype(d[col])]

for col in text_cols(df):
    df[col] = df[col].where(df[col].isna(), df[col].astype(str).str.strip())
df = df.replace(['?', 'NA', 'N/A', 'na', 'n/a', 'NaN', 'nan', 'null', 'NULL', 'None', '-', '--', ''], np.nan)
for col in text_cols(df):
    conv = pd.to_numeric(df[col], errors='coerce')
    if df[col].notna().sum() > 0 and conv.notna().sum() >= 0.9 * df[col].notna().sum():
        df[col] = conv
for col in df.select_dtypes(include='bool').columns:
    df[col] = df[col].astype(str)

txt       = text_cols(df)
num_cols  = df.select_dtypes(include=np.number).columns.tolist()
cont_cols = [col for col in num_cols if df[col].nunique() > 10] or num_cols
cat_cols  = [col for col in df.columns
             if 2 <= df[col].nunique() <= 10 and (col in txt or col not in cont_cols)]
two_lvl   = [col for col in cat_cols if df[col].nunique() == 2]

def pick(user, default, name):
    if user is None:
        return default
    if user not in df.columns:
        raise KeyError(f"{name}: column '{user}' not found. Available: {list(df.columns)}")
    return user

s_num = pick(NUM_COL, cont_cols[0] if cont_cols else None, 'NUM_COL')
s_cat = pick(CAT_COL, (two_lvl or cat_cols or [None])[0], 'CAT_COL')
s_grp = pick(GROUP_COL, (two_lvl or cat_cols or [None])[0], 'GROUP_COL')
s_gnum = pick(GROUP_NUM, s_num, 'GROUP_NUM')

# default correlation pair = most strongly correlated pair of continuous columns
best_pair = (None, None)
if len(cont_cols) >= 2:
    cm = df[cont_cols].corr().abs().where(~np.eye(len(cont_cols), dtype=bool))
    best_pair = cm.stack().idxmax()
s_cx = pick(CORR_X, best_pair[0], 'CORR_X')
s_cy = pick(CORR_Y, best_pair[1], 'CORR_Y')

print('Continuous numeric columns:', cont_cols)
print('Categorical columns       :', cat_cols)
print('\nAuto-selected (change in CONFIG if needed):')
print(f'  Mean / median / std / custom : {s_num}')
print(f'  Proportion                   : {s_cat}')
print(f'  Group comparison             : {s_gnum} by {s_grp}')
print(f'  Correlation / slope          : {s_cx} vs {s_cy}')
print(f'\nConfidence level = {CONF_LEVEL}, resamples B = {B}')

## Step 4: What is the Bootstrap?
We only have **one sample**, so we cannot see how a statistic would vary from sample to sample. The bootstrap **treats the sample as the population**: draw $n$ observations **with replacement** from the sample (a *resample*), compute the statistic, repeat $B$ times. The spread of these $B$ values approximates the **sampling distribution** of the statistic, which gives its standard error and confidence interval - **without distributional assumptions**.

1. Take sample of size $n$  ->  2. resample $n$ values with replacement  ->  3. compute statistic  ->  4. repeat $B$ times  ->  5. take percentiles of the $B$ statistics

In [ ]:
if s_num is None:
    print('No numeric column available')
else:
    x_demo = df[s_num].dropna().astype(float).values
    rng_demo = np.random.default_rng(SEED)
    print(f'Original sample of {s_num} (n = {len(x_demo)}): first 8 values = {np.round(x_demo[:8], 2)}')
    print(f'Original sample mean = {x_demo.mean():.4f}\n')
    for i in range(1, 4):
        res = rng_demo.choice(x_demo, size=len(x_demo), replace=True)
        print(f'Resample {i}: first 8 values = {np.round(res[:8], 2)} ...  mean = {res.mean():.4f}')
    print('\nNotice: values repeat (sampling WITH replacement) and each resample gives a slightly different mean.')

## Bootstrap Engine (used by all steps below)

In [ ]:
ALPHA = 1 - CONF_LEVEL
Z = stats.norm.ppf(1 - ALPHA / 2)

def bootstrap(n, stat_vec, B=B, seed=SEED):
    """Draws B resamples of the row indices 0..n-1 and returns the B statistic values.
       stat_vec receives an index matrix of shape (b, n) and returns b statistic values."""
    rng = np.random.default_rng(seed)
    chunk = max(1, int(2e6 // n))                    # process in chunks to save memory
    out, left = [], B
    while left > 0:
        b = min(chunk, left)
        out.append(np.asarray(stat_vec(rng.integers(0, n, size=(b, n)))))
        left -= b
    out = np.concatenate(out)
    return out[np.isfinite(out)]

def jackknife(n, stat_one, max_n=3000, seed=0):
    """Leave-one-out values of the statistic (needed for the BCa acceleration constant)."""
    idx_all = np.arange(n)
    pick_idx = idx_all if n <= max_n else np.random.default_rng(seed).choice(n, max_n, replace=False)
    return np.array([stat_one(np.delete(idx_all, i)) for i in pick_idx])

def all_cis(boot, theta, jack=None):
    """Percentile, Basic, Normal and (if jackknife given) BCa confidence intervals."""
    lo, hi = np.percentile(boot, [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)])
    res = {'Percentile': (lo, hi),
           'Basic (reverse percentile)': (2 * theta - hi, 2 * theta - lo)}
    se = boot.std(ddof=1)
    res['Normal (estimate +/- z*SE)'] = (theta - Z * se, theta + Z * se)
    extra = {}
    if jack is not None and len(jack) > 2 and np.isfinite(jack).all():
        p0 = (np.sum(boot < theta) + 0.5 * np.sum(boot == theta)) / len(boot)
        p0 = np.clip(p0, 1 / (len(boot) + 1), 1 - 1 / (len(boot) + 1))
        z0 = stats.norm.ppf(p0)                                       # bias-correction
        jm = jack.mean()
        den = 6 * (np.sum((jm - jack) ** 2)) ** 1.5
        acc = np.sum((jm - jack) ** 3) / den if den > 0 else 0.0      # acceleration (skewness)
        adj = lambda zq: stats.norm.cdf(z0 + (z0 + zq) / (1 - acc * (z0 + zq)))
        q_lo, q_hi = adj(-Z), adj(Z)
        res['BCa (bias-corrected accelerated)'] = tuple(np.percentile(boot, [100 * q_lo, 100 * q_hi]))
        extra = {'z0 (bias)': z0, 'a (acceleration)': acc}
    return res, extra

summary = []       # final table of all intervals

def report(name, theta, boot, jack=None, classical=None, ax=None):
    """Prints the table of CIs, stores them in `summary`, draws the bootstrap distribution."""
    cis, extra = all_cis(boot, theta, jack)
    if classical is not None:
        cis[classical[0]] = classical[1]
    tab = pd.DataFrame(cis, index=['Lower', 'Upper']).T
    tab['Width'] = tab['Upper'] - tab['Lower']
    print(f'{name}: estimate = {theta:.4f} | bootstrap SE = {boot.std(ddof=1):.4f} | '
          f'bias = {boot.mean() - theta:+.4f} | resamples used = {len(boot)}')
    if extra:
        print(f'BCa constants: z0 = {extra["z0 (bias)"]:.4f}, acceleration a = {extra["a (acceleration)"]:.4f}')
    display(tab)
    for meth, (l, u) in cis.items():
        summary.append({'Statistic': name, 'Method': meth, 'Estimate': theta, 'Lower': l, 'Upper': u})

    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(boot, kde=True, ax=ax, color='steelblue')
    ax.axvline(theta, color='black', lw=2, label=f'estimate = {theta:.3f}')
    pl, pu = cis['Percentile']
    ax.axvline(pl, color='red', ls='--', label=f'{CONF_LEVEL*100:.0f}% percentile CI')
    ax.axvline(pu, color='red', ls='--')
    if 'BCa (bias-corrected accelerated)' in cis:
        bl, bu = cis['BCa (bias-corrected accelerated)']
        ax.axvline(bl, color='green', ls=':', lw=2, label='BCa CI'); ax.axvline(bu, color='green', ls=':', lw=2)
    ax.set_title(f'Bootstrap distribution: {name}'); ax.set_xlabel('statistic value'); ax.legend(fontsize=8)
    if own:
        plt.tight_layout(); plt.show()
    return cis

## Step 5: Bootstrap CI for the Mean
Compared with the classical **t-interval** $\bar{x} \pm t_{n-1}\,s/\sqrt{n}$.

In [ ]:
if s_num is None:
    print('No numeric column - skipped')
else:
    x = df[s_num].dropna().astype(float).values
    n = len(x)
    print(f'Column: {s_num} | n = {n} | skewness = {stats.skew(x):.3f}\n')

    theta = x.mean()
    boot_mean = bootstrap(n, lambda idx: x[idx].mean(axis=1))
    jack = jackknife(n, lambda idx: x[idx].mean())
    tc = stats.t.ppf(1 - ALPHA / 2, n - 1)
    se = x.std(ddof=1) / np.sqrt(n)
    cis_mean = report(f'Mean of {s_num}', theta, boot_mean, jack,
                      classical=('Classical t-interval', (theta - tc * se, theta + tc * se)))
    print(f'Classical standard error = {se:.4f}  vs  bootstrap standard error = {boot_mean.std(ddof=1):.4f}')

In [ ]:
# One-line version with SciPy's built-in function (same idea, handy to quote in the exam)
if s_num is not None:
    try:
        r = stats.bootstrap((x,), np.mean, confidence_level=CONF_LEVEL, n_resamples=B,
                            method='BCa', random_state=SEED)
        print(f'scipy.stats.bootstrap  BCa {CONF_LEVEL*100:.0f}% CI for the mean: '
              f'({r.confidence_interval.low:.4f}, {r.confidence_interval.high:.4f}), SE = {r.standard_error:.4f}')
    except Exception as e:
        print('scipy.stats.bootstrap not available in this SciPy version:', e)

## Step 6: Median, Standard Deviation, Trimmed Mean
For these statistics there is **no simple classical formula** - the bootstrap is the practical tool.

In [ ]:
if s_num is None:
    print('No numeric column - skipped')
else:
    funcs = {
        'Median'      : (lambda idx: np.median(x[idx], axis=1),            lambda idx: np.median(x[idx])),
        'Std dev'     : (lambda idx: x[idx].std(axis=1, ddof=1),           lambda idx: x[idx].std(ddof=1)),
        f'Trimmed mean ({int(TRIM*100)}%)': (lambda idx: stats.trim_mean(x[idx], TRIM, axis=1),
                                             lambda idx: stats.trim_mean(x[idx], TRIM)),
    }
    thetas = {'Median': np.median(x), 'Std dev': x.std(ddof=1), f'Trimmed mean ({int(TRIM*100)}%)': stats.trim_mean(x, TRIM)}
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    for ax, (nm, (fv, f1)) in zip(axes, funcs.items()):
        bt = bootstrap(n, fv)
        jk = jackknife(n, f1)
        report(f'{nm} of {s_num}', thetas[nm], bt, jk, ax=ax)
        print()
    plt.tight_layout(); plt.show()

## Step 7: Bootstrap CI for a Proportion
Proportion of observations in one category (coded 1/0). Compared with the classical **Wilson** and **Wald (normal)** intervals.

In [ ]:
if s_cat is None:
    print('No categorical column - skipped')
else:
    col = df[s_cat].dropna().astype(str)
    level = str(POS_LEVEL) if POS_LEVEL is not None else col.value_counts().index[0]
    if POS_LEVEL is None:
        print(f'NOTE: POS_LEVEL not set -> using the most frequent category "{level}". Set POS_LEVEL in CONFIG for another category.')
    ind = (col == level).astype(float).values
    nn = len(ind)
    p_hat = ind.mean()
    print(f'{s_cat} = "{level}": {int(ind.sum())} out of {nn}  ->  sample proportion = {p_hat:.4f}\n')

    boot_p = bootstrap(nn, lambda idx: ind[idx].mean(axis=1))
    jk = jackknife(nn, lambda idx: ind[idx].mean())
    # classical intervals
    se_w = np.sqrt(p_hat * (1 - p_hat) / nn)
    centre = (p_hat + Z**2 / (2 * nn)) / (1 + Z**2 / nn)
    half = Z * np.sqrt(p_hat * (1 - p_hat) / nn + Z**2 / (4 * nn**2)) / (1 + Z**2 / nn)
    cis_p = report(f'Proportion of {s_cat}="{level}"', p_hat, boot_p, jk,
                   classical=('Classical Wilson interval', (centre - half, centre + half)))
    print(f'Classical Wald (normal approx.) interval: ({p_hat - Z*se_w:.4f}, {p_hat + Z*se_w:.4f})')
    if nn * p_hat < 10 or nn * (1 - p_hat) < 10:
        print('NOTE: n*p or n*(1-p) < 10 -> the normal approximation is poor; bootstrap/Wilson are safer.')

## Step 8: Difference Between Two Groups
Resample **within each group separately**, compute the difference each time. **If the CI does not contain 0**, the groups differ significantly at level $\alpha = 1 - $ confidence level. Compared with the classical **Welch** interval.

In [ ]:
ok_g = False
if s_gnum is None or s_grp is None:
    print('Need one numeric and one categorical column - skipped')
else:
    dg = df[[s_gnum, s_grp]].dropna().copy()
    dg[s_grp] = dg[s_grp].astype(str)
    if GROUP_LEVELS is not None:
        g1, g2 = str(GROUP_LEVELS[0]), str(GROUP_LEVELS[1])
    else:
        g1, g2 = dg[s_grp].value_counts().index[:2]
        if dg[s_grp].nunique() > 2:
            print(f'NOTE: {s_grp} has {dg[s_grp].nunique()} levels -> comparing the two largest: {g1} vs {g2}. Use GROUP_LEVELS to choose.')
    a = dg.loc[dg[s_grp] == g1, s_gnum].astype(float).values
    b = dg.loc[dg[s_grp] == g2, s_gnum].astype(float).values
    ok_g = len(a) >= 3 and len(b) >= 3
    if not ok_g:
        print('Not enough observations in the groups - skipped')

def boot_two(a, b, stat2, B=B, seed=SEED):
    """Stratified bootstrap: each group is resampled separately. stat2(A, B) works row-wise on 2-D arrays."""
    rng = np.random.default_rng(seed)
    chunk = max(1, int(2e6 // max(len(a), len(b))))
    out, left = [], B
    while left > 0:
        k = min(chunk, left)
        A = a[rng.integers(0, len(a), size=(k, len(a)))]
        Bm = b[rng.integers(0, len(b), size=(k, len(b)))]
        out.append(stat2(A, Bm)); left -= k
    out = np.concatenate(out)
    return out[np.isfinite(out)]

def jack_two(a, b, f, max_n=1500):
    ra = np.random.default_rng(0).choice(len(a), min(len(a), max_n), replace=False)
    rb = np.random.default_rng(1).choice(len(b), min(len(b), max_n), replace=False)
    return np.array([f(np.delete(a, i), b) for i in ra] + [f(a, np.delete(b, j)) for j in rb])

if ok_g:
    print(f'{s_gnum}: group "{g1}" (n = {len(a)}, mean = {a.mean():.3f}) vs group "{g2}" (n = {len(b)}, mean = {b.mean():.3f})\n')

    # --- difference of means ---
    d_hat = a.mean() - b.mean()
    bt = boot_two(a, b, lambda A, Bm: A.mean(axis=1) - Bm.mean(axis=1))
    jk = jack_two(a, b, lambda u, v: u.mean() - v.mean())
    v1, v2 = a.var(ddof=1) / len(a), b.var(ddof=1) / len(b)
    se_d = np.sqrt(v1 + v2)
    dof = (v1 + v2) ** 2 / (v1 ** 2 / (len(a) - 1) + v2 ** 2 / (len(b) - 1))
    tc = stats.t.ppf(1 - ALPHA / 2, dof)
    cis_d = report(f'Mean({g1}) - Mean({g2})', d_hat, bt, jk,
                   classical=('Classical Welch t-interval', (d_hat - tc * se_d, d_hat + tc * se_d)))
    lo, hi = cis_d['Percentile']
    print(f'-> The {CONF_LEVEL*100:.0f}% bootstrap CI ({lo:.3f}, {hi:.3f}) ' +
          ('does NOT contain 0: significant difference between the group means.' if lo > 0 or hi < 0
           else 'CONTAINS 0: no significant difference between the group means.'))

    # --- difference of medians ---
    print()
    m_hat = np.median(a) - np.median(b)
    bt2 = boot_two(a, b, lambda A, Bm: np.median(A, axis=1) - np.median(Bm, axis=1))
    jk2 = jack_two(a, b, lambda u, v: np.median(u) - np.median(v))
    cis_m = report(f'Median({g1}) - Median({g2})', m_hat, bt2, jk2)
    lo, hi = cis_m['Percentile']
    print(f'-> Median difference: 0 is ' + ('OUTSIDE the interval (significant).' if lo > 0 or hi < 0 else 'INSIDE the interval (not significant).'))

## Step 9: Correlation and Regression Slope (Pairs Bootstrap)
Here whole **rows (x, y) are resampled together**, so the relationship between the two variables is preserved. Compared with the classical Fisher-z interval (correlation) and the t-interval (slope).

In [ ]:
if s_cx is None or s_cy is None or s_cx == s_cy:
    print('Need two different numeric columns - skipped')
else:
    dp = df[[s_cx, s_cy]].dropna().astype(float)
    px, py = dp[s_cx].values, dp[s_cy].values
    npair = len(px)
    print(f'Pair: x = {s_cx}, y = {s_cy} | n = {npair}\n')

    def corr_vec(idx):
        xs, ys = px[idx], py[idx]
        xm, ym = xs - xs.mean(1, keepdims=True), ys - ys.mean(1, keepdims=True)
        return (xm * ym).sum(1) / np.sqrt((xm ** 2).sum(1) * (ym ** 2).sum(1))

    def slope_vec(idx):
        xs, ys = px[idx], py[idx]
        xm, ym = xs - xs.mean(1, keepdims=True), ys - ys.mean(1, keepdims=True)
        return (xm * ym).sum(1) / (xm ** 2).sum(1)

    # --- Pearson correlation ---
    r = np.corrcoef(px, py)[0, 1]
    z_r, se_z = np.arctanh(r), 1 / np.sqrt(npair - 3)
    fisher = (np.tanh(z_r - Z * se_z), np.tanh(z_r + Z * se_z))
    bt = bootstrap(npair, corr_vec)
    jk = jackknife(npair, lambda idx: np.corrcoef(px[idx], py[idx])[0, 1])
    cis_r = report(f'Correlation({s_cx}, {s_cy})', r, bt, jk, classical=('Classical Fisher-z interval', fisher))
    lo, hi = cis_r['Percentile']
    print('-> ' + ('The interval excludes 0: the correlation is significant.' if lo > 0 or hi < 0 else 'The interval contains 0: no significant correlation.'))

    # --- regression slope ---
    print()
    lr = stats.linregress(px, py)
    tc = stats.t.ppf(1 - ALPHA / 2, npair - 2)
    bt = bootstrap(npair, slope_vec)
    jk = jackknife(npair, lambda idx: stats.linregress(px[idx], py[idx]).slope)
    cis_s = report(f'Slope of {s_cy} on {s_cx}', lr.slope, bt, jk,
                   classical=('Classical t-interval (OLS)', (lr.slope - tc * lr.stderr, lr.slope + tc * lr.stderr)))

    plt.figure(figsize=(6, 4.5))
    sns.regplot(x=px, y=py, scatter_kws={'alpha': 0.4, 's': 15}, line_kws={'color': 'red'}, ci=int(CONF_LEVEL * 100), n_boot=500)
    plt.xlabel(s_cx); plt.ylabel(s_cy); plt.title('Scatter + fitted line (shaded band = bootstrap CI of the line)')
    plt.show()

## Step 10: Bootstrap CI for ANY Statistic
This is the main strength of the bootstrap - plug in **any function** of the data (set `CUSTOM_STAT` in CONFIG, e.g. `lambda a: np.percentile(a, 90)`, `lambda a: np.mean(a > 50)`, interquartile range...).

In [ ]:
if s_num is None:
    print('No numeric column - skipped')
else:
    if CUSTOM_STAT is None:
        stat_fn, stat_name = (lambda a: a.std(ddof=1) / a.mean()), f'Coefficient of variation of {s_num} (std/mean)'
        print('CUSTOM_STAT not set -> demonstrating with the coefficient of variation.\n')
    else:
        stat_fn, stat_name = CUSTOM_STAT, f'Custom statistic of {s_num}'
    theta = stat_fn(x)
    bt = bootstrap(n, lambda idx: np.apply_along_axis(stat_fn, 1, x[idx]))
    jk = jackknife(n, lambda idx: stat_fn(x[idx]))
    report(stat_name, theta, bt, jk)

## Step 11: How Many Resamples (B) Are Needed?
The CI endpoints are themselves estimated from $B$ resamples, so they wobble for small $B$. Each dot below is one repetition (different seed).

In [ ]:
if s_num is not None:
    B_list = [50, 100, 300, 1000, 3000, 10000]
    rows = []
    for bb in B_list:
        for rep in range(5):
            bt = bootstrap(n, lambda idx: x[idx].mean(axis=1), B=bb, seed=1000 * rep + bb)
            lo, hi = np.percentile(bt, [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)])
            rows.append([bb, lo, hi])
    conv = pd.DataFrame(rows, columns=['B', 'lower', 'upper'])
    plt.figure(figsize=(8, 4.5))
    plt.scatter(conv['B'], conv['lower'], color='tomato', alpha=0.7, label='lower limit')
    plt.scatter(conv['B'], conv['upper'], color='seagreen', alpha=0.7, label='upper limit')
    plt.xscale('log'); plt.xlabel('B (number of resamples, log scale)'); plt.ylabel('CI limit')
    plt.title(f'Percentile CI for the mean of {s_num}: stability vs B'); plt.legend()
    plt.show()
    print(conv.groupby('B').agg(lower_sd=('lower', 'std'), upper_sd=('upper', 'std')))
    print('\nThe spread between repetitions shrinks as B grows -> use B >= 1000 for intervals (>= 10000 for BCa / 99% CI).')

## Step 12: Coverage Simulation - Does a 95% CI Really Capture the True Value 95% of the Time?
We treat your column as the **whole population** (true mean known). We repeatedly draw a **small sample**, build the percentile-bootstrap CI and the classical t-CI, and count how often each contains the true mean.

In [ ]:
if RUN_COVERAGE and s_num is not None and len(x) >= 20:
    pop = x
    mu = pop.mean()
    n0 = int(min(20, len(pop) // 2))
    SIMS, B_SIM = 300, 1000
    rng = np.random.default_rng(SEED)
    cov = {'Bootstrap percentile': 0, 'Classical t': 0}
    wid = {'Bootstrap percentile': [], 'Classical t': []}
    for s in range(SIMS):
        smp = rng.choice(pop, n0, replace=False)
        bt = smp[rng.integers(0, n0, size=(B_SIM, n0))].mean(axis=1)
        lo, hi = np.percentile(bt, [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)])
        cov['Bootstrap percentile'] += (lo <= mu <= hi); wid['Bootstrap percentile'].append(hi - lo)
        tc_ = stats.t.ppf(1 - ALPHA / 2, n0 - 1); se_ = smp.std(ddof=1) / np.sqrt(n0)
        cov['Classical t'] += (smp.mean() - tc_ * se_ <= mu <= smp.mean() + tc_ * se_); wid['Classical t'].append(2 * tc_ * se_)
    print(f'Population mean = {mu:.4f} | sample size n = {n0} | simulations = {SIMS} | nominal level = {CONF_LEVEL*100:.0f}%\n')
    display(pd.DataFrame({'Coverage (%)': {k: 100 * v / SIMS for k, v in cov.items()},
                          'Average width': {k: np.mean(v) for k, v in wid.items()}}))
    print('Coverage close to the nominal level = the method is reliable. Bootstrap intervals tend to be slightly too narrow for very small n.')
else:
    print('Coverage simulation skipped (RUN_COVERAGE = False, no numeric column, or fewer than 20 rows)')

## Step 13: Summary of All Intervals

In [ ]:
summary_df = pd.DataFrame(summary)
if len(summary_df):
    pivot = summary_df[summary_df['Method'].isin(['Percentile', 'BCa (bias-corrected accelerated)'])]
    display(pivot.round(4))
    summary_df.to_csv('bootstrap_confidence_intervals.csv', index=False)
    print('Saved:', os.path.join(os.getcwd(), 'bootstrap_confidence_intervals.csv'))

## Theory Cheat-Sheet (for viva / written answers)

**Bootstrap algorithm:** (1) from a sample of size $n$ draw $n$ values **with replacement**; (2) compute the statistic $\hat\theta^*$; (3) repeat $B$ times; (4) use the $B$ values $\hat\theta^*_1,\dots,\hat\theta^*_B$ to estimate SE, bias and confidence limits.

| Method | Formula | Notes |
|---|---|---|
| **Percentile** | $[\theta^*_{\alpha/2},\ \theta^*_{1-\alpha/2}]$ | Simplest; good when the bootstrap distribution is symmetric and unbiased |
| **Basic (reverse percentile)** | $[2\hat\theta - \theta^*_{1-\alpha/2},\ 2\hat\theta - \theta^*_{\alpha/2}]$ | Reflects the percentiles around the estimate |
| **Normal** | $\hat\theta \pm z_{1-\alpha/2}\, SE_{boot}$ | Assumes the bootstrap distribution is roughly normal |
| **BCa** | percentiles adjusted by bias $z_0$ and acceleration $a$ | Corrects for bias and skewness; **usually the most accurate** |

* **Bootstrap SE** = standard deviation of the $B$ bootstrap statistics. **Bootstrap bias** = mean of bootstrap statistics - original estimate.
* **Interpreting a CI:** a 95% CI means that the *procedure* captures the true parameter in about 95% of repeated samples - **not** "95% probability the parameter is in this particular interval".
* **Testing with a CI:** if the CI for a difference (or correlation/slope) does not contain 0, reject H0 at level $\alpha$.
* **Advantages:** no normality assumption, works for medians, percentiles, ratios, correlations, any statistic; simple to implement.
* **Limitations:** assumes the sample represents the population (poor for tiny samples); fails for extreme statistics (min, max); needs independent observations (use block bootstrap for time series); slightly too narrow for very small $n$.
* **Pairs bootstrap** resamples whole rows (for correlation/regression); **stratified bootstrap** resamples within each group (for group comparisons).
* **Choosing B:** $B \ge 1000$ for standard errors and percentile CIs, $B \ge 10000$ for BCa or 99% intervals.